In [21]:
from sympy.physics.units import temperature
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

In [44]:
#loading env, keys
load_dotenv()

#laoding llm model
# llm = ChatOllama(model="llama3.1:8b")
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite",temperature=0.2)

In [10]:
# Document Laoder
video_id = "7ARBJQn6QkM"
yt_api =YouTubeTranscriptApi()

transcript = yt_api.fetch(video_id)
trans = [doc.text for doc in transcript]
print(trans)

#Cleaning the transcript
transcript_clean = " ".join(doc.text for doc in transcript)
transcript_clean

["At some point, you have to believe something.\xa0\nWe've reinvented computing as we know it. What", 'is the vision for what you see coming next? We\xa0\nasked ourselves, if it can do this, how far can', 'it go? How do we get from the robots that\xa0\nwe have now to the future world that you', 'see? Cleo, everything that moves will be\xa0\nrobotic someday and it will be soon. We', "invested tens of billions of dollars before\xa0\nit really happened. No that's very good, you", 'did some research! But the big breakthrough\xa0\nI would say is when we...', "That's Jensen Huang, and whether you know it or not\nhis decisions are\xa0shaping your future. He's the CEO of", 'NVIDIA, the company that skyrocketed over the past few\nyears\xa0to become one of the most valuable companies in', 'the world because they led a fundamental shift\xa0\nin how computers work unleashing this current', 'explosion of what\'s possible with technology.\xa0\n"NVIDIA\'s done it again!" We found ourselves being', 'o

'At some point, you have to believe something.\xa0\nWe\'ve reinvented computing as we know it. What is the vision for what you see coming next? We\xa0\nasked ourselves, if it can do this, how far can it go? How do we get from the robots that\xa0\nwe have now to the future world that you see? Cleo, everything that moves will be\xa0\nrobotic someday and it will be soon. We invested tens of billions of dollars before\xa0\nit really happened. No that\'s very good, you did some research! But the big breakthrough\xa0\nI would say is when we... That\'s Jensen Huang, and whether you know it or not\nhis decisions are\xa0shaping your future. He\'s the CEO of NVIDIA, the company that skyrocketed over the past few\nyears\xa0to become one of the most valuable companies in the world because they led a fundamental shift\xa0\nin how computers work unleashing this current explosion of what\'s possible with technology.\xa0\n"NVIDIA\'s done it again!" We found ourselves being one of the most important te

In [14]:
# Text Splitting
splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
data_chunks = splitter.split_text(transcript_clean)
len(data_chunks)

118

In [23]:
# Convert to Embeddings -> Call embedding model
embeddings = OllamaEmbeddings(model='embeddinggemma')

In [24]:
# Vector store creation and storing
vc_store = FAISS.from_texts(data_chunks,embeddings)

In [25]:
# Checking Similarity search over vector we created
vc_store.similarity_search(query="What is advancing in Robotics with AI")

[Document(id='8c286188-3acf-4efe-8497-7dd5c6c8b5b5', metadata={}, page_content="I understand, we might be about to see a huge leap in what all of these robots are capable of\xa0\nbecause we're changing how we train them. Up until recently you've either had to train your robot in\xa0\nthe real world where it could get damaged or wear down or you could get data from fairly limited\xa0\nsources like humans in motion capture suits. But that means that robots aren't getting as many\xa0\nexamples as they'd need to learn more quickly. But now we're starting to train robots in digital"),
 Document(id='74081de3-6d72-4197-98a8-8a47fed7dc0b', metadata={}, page_content="be right on robotics and - this is my question. What are the bets you're making now? the latest bet we\njust described at the CES and I'm very very proud of it and I'm very excited about it is the\xa0\nfusion of Omniverse and Cosmos so that we have this new type of generative world generation\xa0\nsystem, this multiverse generation

In [28]:
# Retriever
retriever = vc_store.as_retriever(search_kwrgs={"k":3})

In [29]:
#Testing retriever
retriever.invoke(input="What is advancing in Robotics with AI")

[Document(id='8c286188-3acf-4efe-8497-7dd5c6c8b5b5', metadata={}, page_content="I understand, we might be about to see a huge leap in what all of these robots are capable of\xa0\nbecause we're changing how we train them. Up until recently you've either had to train your robot in\xa0\nthe real world where it could get damaged or wear down or you could get data from fairly limited\xa0\nsources like humans in motion capture suits. But that means that robots aren't getting as many\xa0\nexamples as they'd need to learn more quickly. But now we're starting to train robots in digital"),
 Document(id='74081de3-6d72-4197-98a8-8a47fed7dc0b', metadata={}, page_content="be right on robotics and - this is my question. What are the bets you're making now? the latest bet we\njust described at the CES and I'm very very proud of it and I'm very excited about it is the\xa0\nfusion of Omniverse and Cosmos so that we have this new type of generative world generation\xa0\nsystem, this multiverse generation

In [30]:
#Augementation
#Prompt Template

prompt = PromptTemplate(template="""You are a helpful Youtube brief assistant, Answer from the following context, If the context is insufficient then directly convey that I dont know. {context}, Here is the
Questions{query}""",input_variables=['context',"query"])

In [74]:
#Query
query = "How AI can change future generation?"

In [75]:
#creating Context
context = retriever.invoke(query)
#cleaning the context
cleaned_context = " ".join(i.page_content for i in context)

"shortly after that, computers started to emerge and so we had to ask ourselves how do we use computers\xa0\nto do our jobs better? The next generation doesn't have to ask that question but it has to ask\xa0\nobviously next question, how can I use AI to do my job better? That is start and finish I think\xa0\nfor everybody. It's a really exciting and scary and therefore worthwhile question I think for everyone.\xa0\nI think it's going to be incredibly fun. AI is obviously a word that people are just learning the vision for what you see coming next? In order to talk about this big moment we're in with AI\xa0\nI think we need to go back to video games in the '90s. At the time I know game developers wanted\xa0\nto create more realistic looking graphics but the hardware couldn't keep up with all of that\xa0\nnecessary math. NVIDIA came up with a solution that would change not just games\xa0\nbut computing itself. Could you take us back there and explain what was happening and what an AI tut

In [76]:
#Creating final prompt
final_prompt = prompt.invoke({'query':query,'context':cleaned_context})

In [77]:
#Generation
answer = llm.invoke(final_prompt)

In [78]:
answer.content[0]['text']

'Based on the provided context, AI can change the future generation by:\n\n* Allowing them to ask how to use AI to do their jobs better (unlike past generations who asked how to use computers).\n* Providing an AI tutor that can teach anything, help with programming, writing, analyzing, thinking, and reasoning.\n* Empowering people to become "superhumans" not through innate superpowers, but through "super AIs."'